In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, VotingRegressor
     

train_data= pd.read_csv("train_data.csv")
test_data= pd.read_csv("test_data.csv")

print('Shape of Train dataset: ', train_data.shape)
print('Shape of Test dataset: ', test_data.shape)
     

X= train_data.drop(columns=['purchaseValue'])
y= train_data['purchaseValue']
     

def feature_engineer(data):
    data['date'] = pd.to_datetime(data['date'], format='%Y%m%d')
    data['month'] = data['date'].dt.month
    data['day'] =data['date'].dt.day
    data['day_of_week']=data['date'].dt.dayofweek
    data['weekday']= data['date'].dt.weekday
    data['week_of_year'] = data['date'].dt.isocalendar().week.astype(int)
    data= data.drop(columns=['date'])

    data['sessionStart'] = pd.to_datetime(data['sessionStart'], unit='s')
    data['hour'] =data['sessionStart'].dt.hour
    data = data.drop(columns=['sessionStart'])

    return data
     

X = feature_engineer(X)
     

final_features = [
    'sessionNumber',
    'trafficSource.adwordsClickInfo.page',
    'pageViews',
    'locationZone', 'totals.bounces',
    'new_visits',
    'month', 'day_of_week',
    'hour', 'day',]

X_final = X[final_features]
     

pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('etr', ExtraTreesRegressor(random_state=39, n_jobs=-1))])
     

X_train, X_val, y_train, y_val = train_test_split(X_final, y, test_size=0.2, random_state=39)
     

pipeline.fit(X_train, y_train)
print("Train r2 score:", pipeline.score(X_train, y_train))
print("Validation score:", pipeline.score(X_val, y_val))
     

Shape of Train dataset:  (116023, 52)
Shape of Test dataset:  (29006, 51)
Train r2 score: 0.9994921582209473
Validation score: 0.5852676336199688
